# Day 04 — Streaming, Quality and Trust

**Student:** Danah Almudaifer  
**Project:** Masar Mini-Lakehouse  
**Programme:** Modern Data Engineering for AI Systems (SDA-DSC-214)

Covers **LAB 05** and **LAB 06**. The cells below keep the successful Kafka/Structured Streaming and Great Expectations evidence while removing intermediate troubleshooting attempts.

This notebook preserves the executed evidence from my completed Colab run. The project uses only the supplied synthetic Masar dataset.


In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.3.3,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 "
    "pyspark-shell"
)

print(os.environ["PYSPARK_SUBMIT_ARGS"])

--packages io.delta:delta-spark_2.12:3.3.3,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 pyspark-shell


In [42]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lo_ncbvo


In [45]:
import subprocess
import sys
import shlex

pip_args = shlex.split('install -q kafka-python==2.2.15')
subprocess.run(
    [sys.executable, '-m', 'pip', *pip_args],
    check=True,
)


In [53]:
import subprocess

bash_script = 'set -e\n\nKAFKA_VERSION="4.0.2"\nSCALA_VERSION="2.13"\n\nKAFKA_HOME="/content/kafka_${SCALA_VERSION}-${KAFKA_VERSION}"\nKAFKA_TGZ="/content/kafka_${SCALA_VERSION}-${KAFKA_VERSION}.tgz"\nKAFKA_CONFIG="/content/masar-kafka.properties"\nKAFKA_DATA="/content/masar-kafka-data"\nKAFKA_LOG="/content/masar-kafka.log"\n\necho "=== Java ==="\njava -version\n\n# Download Kafka only if it is not already downloaded\nif [ ! -d "$KAFKA_HOME" ]; then\n    echo "Downloading Kafka ${KAFKA_VERSION}..."\n\n    wget -q \\\n      "https://archive.apache.org/dist/kafka/${KAFKA_VERSION}/kafka_${SCALA_VERSION}-${KAFKA_VERSION}.tgz" \\\n      -O "$KAFKA_TGZ"\n\n    tar -xzf "$KAFKA_TGZ" -C /content\nfi\n\necho "Kafka home: $KAFKA_HOME"\n\n# Create a Colab-specific configuration\ncp "$KAFKA_HOME/config/server.properties" "$KAFKA_CONFIG"\n\n# Keep Kafka data inside /content\nsed -i "s#^log.dirs=.*#log.dirs=$KAFKA_DATA#" "$KAFKA_CONFIG"\n\n# Clean previous incomplete Kafka data\nrm -rf "$KAFKA_DATA"\n\n# Create KRaft cluster\nKAFKA_CLUSTER_ID="$("$KAFKA_HOME/bin/kafka-storage.sh" random-uuid)"\n\necho "Cluster ID: $KAFKA_CLUSTER_ID"\n\n"$KAFKA_HOME/bin/kafka-storage.sh" format \\\n    --standalone \\\n    -t "$KAFKA_CLUSTER_ID" \\\n    -c "$KAFKA_CONFIG"\n\n# Start Kafka in background\nnohup "$KAFKA_HOME/bin/kafka-server-start.sh" \\\n    "$KAFKA_CONFIG" \\\n    > "$KAFKA_LOG" 2>&1 &\n\necho "Kafka process started."'

subprocess.run(
    ['bash', '-lc', bash_script],
    check=True,
)


=== Java ===
Kafka home: /content/kafka_2.13-4.0.2
Cluster ID: 72jFXUcBSOaRiTjmdJbrhw
Formatting dynamic metadata voter directory /content/masar-kafka-data with metadata.version 4.0-IV3.
Kafka process started.


openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment (build 17.0.20+8-1-22.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20+8-1-22.04-Ubuntu, mixed mode, sharing)


In [54]:
import time
import socket

time.sleep(8)

def kafka_is_running():
    try:
        with socket.create_connection(
            ("127.0.0.1", 9092),
            timeout=3
        ):
            return True
    except OSError:
        return False

print("Kafka listening on 127.0.0.1:9092:", kafka_is_running())

Kafka listening on 127.0.0.1:9092: True


In [61]:
import json

from masar.streaming import stream_preflight

preflight = stream_preflight()

print(json.dumps(preflight, indent=2))

{
  "scope": "LOCAL_DEPENDENCY_CHECK_ONLY",
  "kafka_executed": false,
  "client_required": "2.2.15",
  "client_observed": "2.2.15",
  "bootstrap_servers": "127.0.0.1:9092",
  "tcp_listening": true,
  "issues": []
}


In [3]:
import importlib.metadata

print(importlib.metadata.version("kafka-python"))

2.2.15


In [59]:
import uuid

from kafka.admin import KafkaAdminClient, NewTopic

test_topic = "masar-day04-" + uuid.uuid4().hex

print("Creating:", test_topic)

admin = KafkaAdminClient(
    bootstrap_servers="127.0.0.1:9092",
    client_id="masar-admin-test",
    request_timeout_ms=15000,
    api_version_auto_timeout_ms=8000
)

try:
    admin.create_topics(
        [
            NewTopic(
                name=test_topic,
                num_partitions=2,
                replication_factor=1
            )
        ],
        timeout_ms=15000
    )

    print("✅ kafka-python create_topics works")

finally:
    admin.close()

Creating: masar-day04-b2317eee38bf47faa9aba9c43394f7fb
✅ kafka-python create_topics works


In [60]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY04_NATIVE_STREAMING",
  "checks": {
    "phase_transport_counts": true,
    "phase_event_counts": true,
    "transport_keys_always_unique": true,
    "restart_same_query_identity": true,
    "restart_new_execution_ids": true,
    "checkpoint_same_for_all_phases": true,
    "actual_checkpoint_files_present": true,
    "producer_consumer_offsets_reconcile": true,
    "source_json_text_preserved": true,
    "event_content_matches_source": true,
    "unique_events_delta_readback": true,
    "all_events_link_to_trusted_trips": true,
    "late_event_retained": true
  }
}
Transport rows: [216, 216, 218, 219]
Unique event IDs: [216, 216, 216, 217]
+-----------+---------+-------------------------+
|event_id   |trip_id  |event_ts                 |
+-----------+---------+-------------------------+
|SYN_E0001_0|SYN_T0001|2026-06-01T06:00:00+03:00|
|SYN_E0001_1|SYN_T0001|2026-06-01T06:04:00+03:00|
|SYN_E0001_2|SYN_T0001|2026-06-01T06:08:00+03:00|
|SYN_E0002_0|SYN_T0002|2026-06-01T0

In [65]:
import subprocess
import sys
import shlex

pip_args = shlex.split('install -q --upgrade      great-expectations==1.7.0      pandas==2.2.3')
subprocess.run(
    [sys.executable, '-m', 'pip', *pip_args],
    check=True,
)


In [66]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "trusted_and_rechecked_pass_gx": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "failed_candidate_not_promoted": true,
    "quarantine_delta_readback": true,
    "approved_readback_same_business_contents": true,
    "source_silver_untouched": true,
    "data_docs_exist_for_all_three_cases": true
  }
}
Quarantined records:
+-------------+----------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------------------------------------------------------------+
|candidate_row|trip_id   |reason_codes       |raw_business_json                             

In [67]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day04_handoff.zip
